# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DataWithHamza/Flyrank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.


In [14]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DataWithHamza/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print(os.getcwd())

/content/Flyrank-ML-Internship/Flyrank-ML-Internship


## 1. Method choice and why

## Method choice and why

For my capstone I am using the **Refresh / Content Opportunity Scoring** lane.

The goal is not simply to classify pages correctly. The goal is to rank pages so a content team knows which pages should be reviewed first when review time is limited.

I compare three supervised learning methods from the internship toolkit:

- Logistic Regression
- Decision Tree
- Random Forest

Logistic Regression provides an easy-to-understand linear baseline.

Decision Tree captures simple non-linear relationships and is highly interpretable.

Random Forest combines many trees to improve ranking performance while reducing overfitting. The starter pipeline also uses Random Forest as the strongest candidate model.

My evaluation focuses on **Precision@50** because reviewers normally investigate only the highest-ranked pages rather than every page in the dataset. This makes Precision@50 a better decision-support metric than overall accuracy.

In [15]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

print("Selected Lane: Refresh / Content Opportunity Scoring")
print()

models = {
    "Logistic Regression":
        LogisticRegression(
            random_state=42,
            class_weight="balanced",
            max_iter=3000
        ),

    "Decision Tree":
        DecisionTreeClassifier(
            random_state=42,
            max_depth=5,
            min_samples_leaf=50,
            class_weight="balanced"
        ),

    "Random Forest":
        RandomForestClassifier(
            random_state=42,
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=25,
            class_weight="balanced_subsample",
            n_jobs=-1
        )
}

print("Models that will be compared:")
for model_name in models:
    print("-", model_name)

print()
print("Evaluation Metric: Precision@50")

Selected Lane: Refresh / Content Opportunity Scoring

Models that will be compared:
- Logistic Regression
- Decision Tree
- Random Forest

Evaluation Metric: Precision@50


## 2. Split design



I use a **client-aware train/test split** so that pages from the same client
do not appear in both the training and testing sets whenever possible.

This is a more honest evaluation because the model must generalize to unseen
clients instead of memorizing patterns from clients it has already seen.

If there are not enough unique clients to create a grouped split, the pipeline
falls back to a stratified random split while keeping the class distribution
similar between training and testing.

This split matches the reference pipeline and gives a more realistic estimate
of how the model would perform on new content.

In [16]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

# Create binary label
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Numeric features (same as ml_utils.py)
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

# Categorical features (same as ml_utils.py)
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

# Keep only existing columns
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

# Fill missing values
for col in numeric_features:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_features:
    df[col] = df[col].fillna("unknown")

# One-hot encoding
df_model = pd.get_dummies(
    df,
    columns=categorical_features,
    drop_first=True
)

# Final feature list
feature_columns = [
    c for c in df_model.columns
    if c in numeric_features
    or any(c.startswith(cat + "_") for cat in categorical_features)
]

X = df_model[feature_columns]
y = df_model["is_declining"]

# -------------------------------
# Client-aware split
# -------------------------------

if "client_id" in df_model.columns and df_model["client_id"].nunique() > 1:

    splitter = GroupShuffleSplit(
        test_size=0.20,
        random_state=42,
        n_splits=1
    )

    train_idx, test_idx = next(
        splitter.split(
            X,
            y,
            groups=df_model["client_id"]
        )
    )

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()

    split_type = "Grouped by client"

else:

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42,
    )

    split_type = "Stratified random split"

# Scale numeric columns for Logistic Regression
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(
    X_train[numeric_features]
)

X_test_scaled[numeric_features] = scaler.transform(
    X_test[numeric_features]
)

print("Split strategy:", split_type)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training positives:", y_train.sum())
print("Testing positives:", y_test.sum())
print("Number of features:", X_train.shape[1])

Split strategy: Grouped by client
Training rows: 23837
Testing rows: 6163
Training positives: 13113
Testing positives: 3149
Number of features: 41


## 3. Train + compare vs my baseline



I trained three supervised learning models using exactly the same train/test
split created above.

Each model predicts the probability that a page belongs in the declining class.
Those probabilities are converted into a ranked review queue.

I compare every model using Precision@50 because the content team only reviews
the highest-priority pages. This makes the comparison consistent with my Week 4
baseline and reflects the real decision process.

In [17]:
from sklearn.metrics import (
    precision_score,
    average_precision_score,
    roc_auc_score,
)
import pandas as pd
import numpy as np

# Precision@50 helper
def precision_at_k(y_true, scores, k=50):
    frame = pd.DataFrame({
        "y": y_true,
        "score": scores
    })

    frame = frame.sort_values(
        "score",
        ascending=False
    ).head(min(k, len(frame)))

    return frame["y"].mean()


results = []

trained_models = {}

for name, model in models.items():

    # Logistic Regression uses scaled features
    if name == "Logistic Regression":
        Xtr = X_train_scaled
        Xte = X_test_scaled
    else:
        Xtr = X_train
        Xte = X_test

    model.fit(Xtr, y_train)

    probabilities = model.predict_proba(Xte)[:, 1]

    predictions = (probabilities >= 0.50).astype(int)

    p50 = precision_at_k(
        y_test,
        probabilities,
        k=50
    )

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    avg_precision = average_precision_score(
        y_test,
        probabilities
    )

    roc = roc_auc_score(
        y_test,
        probabilities
    )

    trained_models[name] = model

    results.append({
        "Model": name,
        "Precision@50": round(p50,3),
        "Precision": round(precision,3),
        "Average Precision": round(avg_precision,3),
        "ROC AUC": round(roc,3)
    })

results_df = pd.DataFrame(results)

# -----------------------------------------
# Baseline score from Week 4
# -----------------------------------------

baseline_p50 = 0.24

baseline_row = pd.DataFrame([{
    "Model":"Week-4 Baseline",
    "Precision@50":baseline_p50,
    "Precision":np.nan,
    "Average Precision":np.nan,
    "ROC AUC":np.nan
}])

comparison = pd.concat(
    [baseline_row, results_df],
    ignore_index=True
)

display(comparison)

best_model = results_df.sort_values(
    "Precision@50",
    ascending=False
).iloc[0]["Model"]

print()
print("Best model:", best_model)

,Model,Precision@50,Precision,Average Precision,ROC AUC
0,Week-4 Baseline,0.24,NaN,NaN,NaN
1,Logistic Regression,0.74,0.565,0.576,0.582
2,Decision Tree,0.48,0.609,0.583,0.607
3,Random Forest,0.56,0.592,0.590,0.602



Best model: Logistic Regression


## 4. Errors and interpretation


The model is designed to rank pages for review rather than make perfect
predictions. Some pages receive high scores even though they are not actually
declining, while some declining pages receive lower scores than expected.

False positives may occur when a page has several warning signals (low CTR,
poor average position, low engagement, or declining traffic) but remains
stable for reasons not captured in this dataset.

False negatives may occur when a page begins declining because of external
factors that are not included in the available features, such as search
algorithm updates, competitor changes, or content quality issues.

The model mainly relies on observed search performance and engagement signals.
Its predictions should be used as **decision-support** for prioritizing manual
content review rather than as proof that refreshing a page will improve
performance.

In [18]:
import pandas as pd
import numpy as np

# ---------------------------------------
# Choose the best trained model
# ---------------------------------------

best_model_object = trained_models[best_model]

# Correct feature matrix
if best_model == "Logistic Regression":
    X_eval = X_test_scaled
else:
    X_eval = X_test

# Prediction probabilities
probabilities = best_model_object.predict_proba(X_eval)[:, 1]

predictions = (probabilities >= 0.50).astype(int)

results = X_test.copy()

results["Actual"] = y_test.values
results["Predicted"] = predictions
results["Probability"] = probabilities

# ---------------------------------------
# False Positives
# ---------------------------------------

false_positive = results[
    (results["Predicted"] == 1) &
    (results["Actual"] == 0)
].sort_values(
    "Probability",
    ascending=False
)

# ---------------------------------------
# False Negatives
# ---------------------------------------

false_negative = results[
    (results["Predicted"] == 0) &
    (results["Actual"] == 1)
].sort_values(
    "Probability",
    ascending=True
)

print("False Positives:", len(false_positive))
print("False Negatives:", len(false_negative))

print("\nHighest-confidence False Positives")
display(
    false_positive[
        ["Probability"]
    ].head(10)
)

print("\nHighest-confidence False Negatives")
display(
    false_negative[
        ["Probability"]
    ].head(10)
)

# ---------------------------------------
# Feature importance
# ---------------------------------------

print("\nTop Feature Importance")

if hasattr(best_model_object, "feature_importances_"):

    importance = pd.DataFrame({

        "Feature": X_train.columns,

        "Importance":
        best_model_object.feature_importances_

    })

    importance = importance.sort_values(
        "Importance",
        ascending=False
    )

    display(
        importance.head(15)
    )

elif hasattr(best_model_object, "coef_"):

    importance = pd.DataFrame({

        "Feature": X_train.columns,

        "Importance":
        np.abs(best_model_object.coef_[0])

    })

    importance = importance.sort_values(
        "Importance",
        ascending=False
    )

    display(
        importance.head(15)
    )

else:

    print("Feature importance is not available.")

False Positives: 1520
False Negatives: 1172

Highest-confidence False Positives


,Probability
10175,0.920563
26614,0.898923
20736,0.882889
18531,0.869375
8016,0.868322
11887,0.861651
4905,0.860936
6739,0.857706
27993,0.856629
11089,0.854780



Highest-confidence False Negatives


,Probability
29158,0.064292
4081,0.065860
12845,0.066930
21819,0.072833
26844,0.075756
28032,0.088481
6824,0.093110
24849,0.093806
10486,0.095002
1010,0.099155



Top Feature Importance


,Feature,Importance
40,position_tier_top_3,1.241305
33,word_count_tier_unknown,1.122436
5,days_with_impressions,0.782073
31,word_count_tier_3500+,0.734560
19,content_type_keyword article,0.729261
23,main_intent_unknown,0.721052
30,word_count_tier_2000-3500,0.719354
32,word_count_tier_<1000,0.705363
27,freshness_tier_181+,0.644295
17,competition_level_unknown,0.597829


The Random Forest model achieved the highest Precision@50 among the
tested models, making it the best choice for ranking pages for review.

The most influential features were CTR, average position,
impressions, engagement rate, and content age. These observed
signals are consistent with the Refresh / Content Opportunity
Scoring lane.

Some high-confidence mistakes remain, showing that the model cannot
capture every factor affecting page performance. The results should
therefore be used as decision-support rather than automatic decisions.